# ServiceTrack: Job Tracking & Customer Visit Analytics Pipeline
### Medallion Architecture (Bronze -> Silver -> Gold) | PySpark - Delta Lake - Auto Loader

**Domain:** Field Service Operations Analytics
**Volume:** ~1,510 service jobs, 300 customers, 43 devices, 90-day operational window

This notebook ingests raw service-center exports (`customers.csv`, `devices.csv`,
`service_jobs.csv`), cleans and enriches them, and produces three business-ready
Gold tables: SLA performance, technician efficiency, and device failure trends.

**Notebook sections**
1. Imports
2. Configuration
3. Explicit Schemas
4. Bronze Layer (Auto Loader ingestion)
5. Silver Layer (cleansing + enrichment)
6. Gold Layer (business aggregates)
7. Optimizations (OPTIMIZE / ZORDER / VACUUM)
8. Validation (data quality gate)
9. Final Summary

## 1. Imports
Only the modules actually used are imported. `pyspark.sql.functions` is aliased
as `F` (the de-facto convention) so transformation code is grep-able and doesn't
collide with Python builtins (`sum`, `min`, `max`, `round`).

In [0]:
import logging
import sys
import time
from dataclasses import dataclass
from typing import Dict, List, Optional

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, StructField, StructType, DateType, DecimalType, IntegerType,
)
from delta.tables import DeltaTable

spark: SparkSession = spark  # noqa: F821 - injected by the Databricks runtime

## 2. Configuration
Every path, threshold, and table name lives in one `PipelineConfig` object instead
of being hard-coded inline. This is what lets the exact same notebook run in
`dev` / `staging` / `prod` by only changing widget values - no code edits, no
risk of a stray hard-coded path shipping to production.

Widgets are used (not just Python variables) so the notebook can also be triggered
as a parameterized Databricks Job/Workflow task.

In [0]:
dbutils.widgets.text("env", "dev", "Environment")  # noqa: F821
dbutils.widgets.text(
    "source_path", "/Volumes/servicetrack/data/landing/source", "Source landing zone"
)  # noqa: F821
dbutils.widgets.text(
    "base_delta_path", "/Volumes/servicetrack/data/landing/delta", "Delta root path"
)  # noqa: F821
dbutils.widgets.text("sla_days", "5", "SLA threshold (days)")  # noqa: F821
dbutils.widgets.text("vacuum_retention_hours", "168", "VACUUM retain hours (default 7 days)")  # noqa: F821

### Environment bootstrap (Unity Catalog Volumes)
This notebook targets Unity Catalog Volumes for all storage, **not** `/mnt/...`
or `/tmp/...`. On Databricks Free Edition (and on any workspace with the
public DBFS root disabled) both of those paths fail with
`[DBFS_DISABLED] Public DBFS root is disabled`. Volumes are unaffected by
that setting and are the supported storage layer going forward, so the cell
below creates the catalog/schema/volume this pipeline needs — idempotently,
so it's safe to re-run.

**You still have to upload the 3 source CSVs yourself** (Catalog Explorer ->
`servicetrack.data.landing` -> "Upload to this volume") — a notebook cell
cannot reach into your laptop's filesystem. The validation cell right after
this one tells you clearly if they're missing instead of failing deep inside
the Bronze layer.

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS servicetrack")
spark.sql("CREATE SCHEMA IF NOT EXISTS servicetrack.data")
spark.sql("CREATE VOLUME IF NOT EXISTS servicetrack.data.landing")

_source_path = dbutils.widgets.get("source_path")  # noqa: F821
try:
    _found_files = {f.name for f in dbutils.fs.ls(_source_path)}  # noqa: F821
except Exception:
    _found_files = set()

_required_files = {"customers.csv", "devices.csv", "service_jobs.csv"}
_missing_files = _required_files - _found_files

if _missing_files:
    print(
        "SETUP INCOMPLETE — the following files are missing from "
        f"{_source_path}: {sorted(_missing_files)}\n\n"
        "Fix: open Catalog (left sidebar) -> servicetrack -> data -> landing, "
        "click the volume, click 'Upload to this volume', and upload "
        "customers.csv, devices.csv, service_jobs.csv there "
        "(into a 'source' subfolder if source_path points at one).\n\n"
        "Re-run this cell once uploaded to confirm, then run the rest of the "
        "notebook."
    )
else:
    print(f"All 3 source files found in {_source_path}. Ready to run Section 4 onward.")

All 3 source files found in /Volumes/servicetrack/data/landing/source. Ready to run Section 4 onward.


In [0]:

@dataclass(frozen=True)
class PipelineConfig:
    """
    Single source of truth for every path, table name, and business threshold
    used across Bronze / Silver / Gold. Frozen (immutable) so a bug elsewhere in
    the notebook can never accidentally mutate a path mid-run.
    """

    env: str
    source_path: str
    base_delta_path: str
    sla_days: int
    vacuum_retention_hours: int

    # ---- Bronze ----
    @property
    def bronze_path(self) -> str:
        return f"{self.base_delta_path}/bronze"

    @property
    def checkpoint_path(self) -> str:
        return f"{self.base_delta_path}/_checkpoints"

    @property
    def schema_location(self) -> str:
        return f"{self.base_delta_path}/_schemas"

    # ---- Silver ----
    @property
    def silver_path(self) -> str:
        return f"{self.base_delta_path}/silver"

    # ---- Gold ----
    @property
    def gold_path(self) -> str:
        return f"{self.base_delta_path}/gold"

    # ---- Table registration (metastore) ----
    catalog_db: str = "servicetrack"

    # Bronze table names
    bronze_customers_tbl: str = "bronze_customers"
    bronze_devices_tbl: str = "bronze_devices"
    bronze_service_jobs_tbl: str = "bronze_service_jobs"

    # Silver table names
    silver_enriched_tbl: str = "enriched_service_jobs"
    silver_quarantine_tbl: str = "quarantined_service_jobs"

    # Gold table names
    gold_sla_tbl: str = "sla_performance_gold"
    gold_technician_tbl: str = "technician_efficiency_gold"
    gold_failure_tbl: str = "failure_trends_gold"


CONFIG = PipelineConfig(
    env=dbutils.widgets.get("env"),  # noqa: F821
    source_path=dbutils.widgets.get("source_path"),  # noqa: F821
    base_delta_path=dbutils.widgets.get("base_delta_path"),  # noqa: F821
    sla_days=int(dbutils.widgets.get("sla_days")),  # noqa: F821
    vacuum_retention_hours=int(dbutils.widgets.get("vacuum_retention_hours")),  # noqa: F821
)

### Logging
A plain `print()`-based notebook is unreviewable when a job fails at 2am. We use
the standard `logging` module with a consistent formatter so every line carries
a timestamp, level, and stage - the output can be piped straight into a log
aggregator (e.g. Azure Log Analytics / Datadog) with zero rework.

In [0]:

def get_logger(name: str = "servicetrack") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:  # avoid duplicate handlers on notebook re-run
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger


log = get_logger()
log.info(f"Pipeline config loaded for env='{CONFIG.env}' | SLA={CONFIG.sla_days} days")

2026-07-15 06:31:25,386 | INFO     | servicetrack | Pipeline config loaded for env='dev' | SLA=5 days


### Reusable utility functions
- `record_count()` wraps `.count()` with logging so every stage prints how many
  rows it produced (a cheap, high-value data quality signal).
- `run_stage()` is a try/except + timing decorator that gives every pipeline
  stage consistent error handling instead of a raw Spark stack trace.

In [0]:

def record_count(df: DataFrame, label: str) -> int:
    n = df.count()
    log.info(f"[{label}] record count = {n:,}")
    return n


def run_stage(stage_name: str):
    """
    Decorator that wraps a pipeline stage function with timing + error handling.
    Any exception is logged with full context before being re-raised, so a
    Databricks Job run shows exactly which stage failed instead of a bare
    stack trace three frames deep in Spark internals.
    """

    def decorator(fn):
        def wrapper(*args, **kwargs):
            start = time.time()
            log.info(f"START stage='{stage_name}'")
            try:
                result = fn(*args, **kwargs)
                elapsed = round(time.time() - start, 2)
                log.info(f"SUCCESS stage='{stage_name}' elapsed={elapsed}s")
                return result
            except Exception as exc:
                elapsed = round(time.time() - start, 2)
                log.error(
                    f"FAILED stage='{stage_name}' elapsed={elapsed}s error={exc}"
                )
                raise RuntimeError(f"Pipeline stage '{stage_name}' failed: {exc}") from exc

        return wrapper

    return decorator


## 3. Explicit Schemas
Auto Loader *can* infer schema, but inference is non-deterministic across runs
(a batch with an all-null column can flip a type) and costs an extra listing
pass over the source data. In production we always pin an explicit `StructType`:
it's faster, deterministic, and doubles as inline documentation of the contract
each source system owes us. Schema **evolution** is still enabled in Section 4
so a genuinely new column doesn't crash the pipeline - it gets merged in safely.

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), nullable=False),
    StructField("customer_name", StringType(), nullable=False),
    StructField("phone_number", StringType(), nullable=False),
    StructField("email", StringType(), nullable=False),
    StructField("city", StringType(), nullable=False),
    StructField("registration_date", DateType(), nullable=False),
])

devices_schema = StructType([
    StructField("device_id", StringType(), nullable=False),
    StructField("brand", StringType(), nullable=False),
    StructField("device_type", StringType(), nullable=False),
    StructField("model_series", StringType(), nullable=False),
    StructField("warranty_months", IntegerType(), nullable=False),
    StructField("price_range", StringType(), nullable=False),
])

# NOTE: date columns are deliberately ingested as StringType in Bronze.
# Bronze's job is to preserve the source byte-for-byte; malformed date strings
# (a real risk with CSV exports from legacy CRMs) should not silently become
# NULL at ingestion time via a DateType parse failure. Typing happens
# explicitly and visibly in Silver (Section 5), where rejects can be logged.
service_jobs_schema = StructType([
    StructField("job_id", StringType(), nullable=True),        # nullable: invalid rows are quarantined, not dropped at read time
    StructField("customer_id", StringType(), nullable=True),
    StructField("device_id", StringType(), nullable=True),
    StructField("issue_type", StringType(), nullable=True),
    StructField("job_status", StringType(), nullable=True),
    StructField("received_date", StringType(), nullable=True),
    StructField("promised_date", StringType(), nullable=True),
    StructField("completed_date", StringType(), nullable=True),
    StructField("technician_id", StringType(), nullable=True),
    StructField("technician_name", StringType(), nullable=True),
    StructField("repair_notes", StringType(), nullable=True),
    StructField("estimated_cost", DecimalType(10, 2), nullable=True),
    StructField("actual_cost", DecimalType(10, 2), nullable=True),
])

## 4. Bronze Layer - Auto Loader Incremental Ingestion

**Why Auto Loader (`cloudFiles`) instead of a plain batch `spark.read`?**
- **Incremental by construction.** Auto Loader tracks which files it has already
  processed via a checkpoint, so a scheduled re-run of this notebook only
  ingests *new* files landing in `source_path` - no manual watermarking, no
  re-scanning the whole directory every run.
- **Cheap file discovery at scale.** It uses cloud-native file notifications /
  efficient listing instead of a full `LIST` on every trigger, which matters
  once a source directory has millions of files (won't show today at 1,510
  rows, but it's the reason this code doesn't need to be rewritten at scale).
- **Schema safety with pinned schemas.** Because we pass an explicit
  `StructType` (Section 3), `cloudFiles.schemaEvolutionMode` is left at its
  default (`none`) — Auto Loader rejects `addNewColumns` when a schema is
  supplied explicitly, since evolution tracking is only meaningful when
  Auto Loader owns schema inference itself. A genuinely new upstream column
  still doesn't crash the stream: it lands in the automatic
  `_rescued_data` column instead of being silently dropped, so nothing is
  lost even though the pinned schema doesn't grow automatically.

Every stream runs with `.trigger(availableNow=True)` - this processes exactly
the files currently sitting in the source and then **stops**, which is the
correct trigger mode for a scheduled batch job (as opposed to a
continuously-running stream, which would hold a cluster up indefinitely).

Every Bronze table also gets three metadata columns: `_ingestion_timestamp`,
`_source_file`, `_load_date` - non-negotiable for auditability. If Gold numbers
ever look wrong, these columns let us answer "which file, ingested when,
produced this row?" in one query.

In [0]:

def ingest_to_bronze(
    file_subpath: str,
    schema: StructType,
    table_name: str,
    file_format: str = "csv",
    options: Optional[Dict[str, str]] = None,
) -> DataFrame:
    """
    Generic Auto Loader ingestion routine, reused for all three Bronze tables.
    Parameterized on (source subpath, schema, table name) so we don't repeat
    the same ~20 lines of boilerplate three times - a change to metadata
    columns or checkpoint layout only has to happen once.
    """
    source_full_path = CONFIG.source_path
    checkpoint = f"{CONFIG.checkpoint_path}/{table_name}"
    schema_loc = f"{CONFIG.schema_location}/{table_name}"
    target_path = f"{CONFIG.bronze_path}/{table_name}"

  
    reader_options = {
        "cloudFiles.format": file_format,
        "cloudFiles.schemaLocation": schema_loc,
        "header": "true",
        "escape": '"',
        "pathGlobFilter": file_subpath,
    }
    if options:
        reader_options.update(options)

    raw_stream = (
        spark.readStream.format("cloudFiles")
        .options(**reader_options)
        .schema(schema)
        .load(source_full_path)
    )

    bronze_df = (
        raw_stream
        .withColumn("_ingestion_timestamp", F.current_timestamp())
     
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_load_date", F.current_date())
    )

    query = (
        bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .outputMode("append")
        .start(target_path)
    )
    query.awaitTermination()

    return spark.read.format("delta").load(target_path)


@run_stage("bronze_ingestion")
def run_bronze_layer() -> Dict[str, DataFrame]:
    bronze_customers = ingest_to_bronze(
        "customers.csv", customers_schema, CONFIG.bronze_customers_tbl
    )
    bronze_devices = ingest_to_bronze(
        "devices.csv", devices_schema, CONFIG.bronze_devices_tbl
    )
    bronze_service_jobs = ingest_to_bronze(
        "service_jobs.csv", service_jobs_schema, CONFIG.bronze_service_jobs_tbl
    )

    record_count(bronze_customers, "bronze_customers")
    record_count(bronze_devices, "bronze_devices")
    record_count(bronze_service_jobs, "bronze_service_jobs")

    return {
        "customers": bronze_customers,
        "devices": bronze_devices,
        "service_jobs": bronze_service_jobs,
    }


bronze_tables = run_bronze_layer()
bronze_customers_df = bronze_tables["customers"]
bronze_devices_df = bronze_tables["devices"]
bronze_service_jobs_df = bronze_tables["service_jobs"]

2026-07-15 06:31:26,199 | INFO     | servicetrack | START stage='bronze_ingestion'
2026-07-15 06:31:44,365 | INFO     | servicetrack | [bronze_customers] record count = 300
2026-07-15 06:31:45,041 | INFO     | servicetrack | [bronze_devices] record count = 43
2026-07-15 06:31:45,661 | INFO     | servicetrack | [bronze_service_jobs] record count = 1,510
2026-07-15 06:31:45,661 | INFO     | servicetrack | SUCCESS stage='bronze_ingestion' elapsed=19.46s


## 5. Silver Layer - Cleansing & Enrichment

This layer does five things, in order:
1. **De-duplicate.** Drop exact duplicate rows, then de-duplicate on `job_id`
   specifically (the source has 10 intentional duplicate `job_id` rows).
2. **Quarantine invalid records.** A row with a NULL `job_id` or NULL
   `customer_id` cannot be joined, aggregated, or traced back to a customer -
   it is written to a separate quarantine table rather than silently dropped,
   so nothing simply vanishes from the audit trail.
3. **Repair `technician_name`.** ~75 rows have a valid `technician_id` but a
   blank `technician_name`. A small technician lookup is built from the rows
   that *do* have both fields, and used to backfill the blanks.
4. **Type dates properly** and derive `repair_duration_days`.
5. **Join** Customers + Devices onto Service Jobs to build one enriched,
   analytics-ready fact table.

### 5.1 De-duplication + Quarantine

In [0]:

def deduplicate_and_quarantine(bronze_jobs: DataFrame) -> Dict[str, DataFrame]:
    """
    Two-step cleansing:
      a) drop fully-identical duplicate rows, then de-dupe on job_id (keeps the
         first occurrence Spark encounters - Bronze carries _ingestion_timestamp
         so "first" is well-defined if the tie-break rule ever needs to change).
      b) split into (valid, quarantined) based on the invalid-record definition:
         job_id IS NULL OR customer_id IS NULL.
    """
    deduped = bronze_jobs.dropDuplicates().dropDuplicates(["job_id"])

    is_invalid = F.col("job_id").isNull() | F.col("customer_id").isNull()

    quarantined_df = deduped.filter(is_invalid).withColumn(
        "_quarantine_reason",
        F.when(F.col("job_id").isNull() & F.col("customer_id").isNull(), F.lit("job_id and customer_id NULL"))
         .when(F.col("job_id").isNull(), F.lit("job_id NULL"))
         .otherwise(F.lit("customer_id NULL")),
    )
    valid_df = deduped.filter(~is_invalid)

    return {"valid": valid_df, "quarantined": quarantined_df}


### 5.2 Technician name backfill
The `technician_id -> technician_name` map is built from the data itself
(rather than hard-coded), so the pipeline stays correct if a 9th technician is
ever onboarded. The lookup is tiny (a handful of rows) - a textbook case for a
**broadcast join**: Spark ships the whole lookup table to every executor
instead of shuffling the multi-thousand-row fact table across the network to
co-locate matching keys. `F.broadcast()` is used explicitly rather than
relying on Spark's auto-broadcast threshold, so the physical plan is correct
by design and doesn't silently regress if `spark.sql.autoBroadcastJoinThreshold`
is ever tightened.

In [0]:

def backfill_technician_name(valid_jobs: DataFrame) -> DataFrame:
    technician_lookup = (
        valid_jobs
        .filter(F.col("technician_id").isNotNull() & F.col("technician_name").isNotNull())
        .select("technician_id", F.col("technician_name").alias("_lookup_name"))
        .dropDuplicates(["technician_id"])
    )

    enriched = (
        valid_jobs.alias("j")
        .join(F.broadcast(technician_lookup).alias("t"), on="technician_id", how="left")
        .withColumn(
            "technician_name",
            F.coalesce(F.col("j.technician_name"), F.col("t._lookup_name")),
        )
        .drop("_lookup_name")
    )
    return enriched


### 5.3 Date typing + repair duration
Every date column arrives as `StringType` from Bronze by design (Section 3).
Here we parse them to `TimestampType` with `to_timestamp`, and derive
`repair_duration_days` using `datediff()`. `completed_date` is legitimately
NULL for `In Progress` / `Pending` / `Cancelled` jobs - `datediff()` against a
NULL operand returns NULL automatically, so no special-casing is strictly
required, but the intent is made explicit with `F.when` for readability and to
guard against a future Spark version changing NULL-propagation semantics.

In [0]:

def type_dates_and_derive_duration(df: DataFrame) -> DataFrame:
    date_typed = (
        df
        .withColumn("received_date", F.to_timestamp("received_date", "yyyy-MM-dd"))
        .withColumn("promised_date", F.to_timestamp("promised_date", "yyyy-MM-dd"))
        .withColumn("completed_date", F.to_timestamp("completed_date", "yyyy-MM-dd"))
    )

    with_duration = date_typed.withColumn(
        "repair_duration_days",
        F.when(
            F.col("completed_date").isNotNull(),
            F.datediff(F.col("completed_date"), F.col("received_date")),
        ).otherwise(F.lit(None).cast(IntegerType())),
    )
    return with_duration


### 5.4 Joins - build the enriched fact table
`customers` (300 rows) and `devices` (43 rows) are both trivially small
relative to `service_jobs` (~1,500 rows). Both are broadcast-joined onto the
job fact table: this avoids shuffling the (larger) fact table entirely - each
executor gets a full local copy of the tiny dimension tables and performs the
join as an in-memory hash lookup. At this row scale it's a minor win in
wall-clock time, but the *pattern* - broadcast the dimension, never the fact -
is exactly what keeps this notebook's join strategy correct after the source
volume grows from 1,500 jobs to 5 million without a single line changing.

In [0]:

def build_enriched_service_jobs(
    jobs_df: DataFrame, customers_df: DataFrame, devices_df: DataFrame
) -> DataFrame:
    customers_slim = customers_df.select(
        "customer_id", "customer_name", "phone_number", "email", "city",
        F.col("registration_date").alias("customer_registration_date"),
    )
    devices_slim = devices_df.select(
        "device_id",
        F.col("brand").alias("device_brand"),
        "device_type",
        "model_series",
        "warranty_months",
        "price_range",
    )

    enriched = (
        jobs_df
        .join(F.broadcast(customers_slim), on="customer_id", how="left")
        .join(F.broadcast(devices_slim), on="device_id", how="left")
        .withColumn(
            "is_delayed",
            F.when(F.col("completed_date").isNull(), F.lit(None).cast("boolean"))
             .otherwise(F.col("repair_duration_days") > F.lit(CONFIG.sla_days)),
        )
        .withColumn("_silver_processed_timestamp", F.current_timestamp())
    )
    return enriched


### 5.5 Orchestrate Silver + write

In [0]:

@run_stage("silver_transformation")
def run_silver_layer(
    bronze_customers: DataFrame, bronze_devices: DataFrame, bronze_jobs: DataFrame
) -> Dict[str, DataFrame]:
    split = deduplicate_and_quarantine(bronze_jobs)
    valid_jobs, quarantined_jobs = split["valid"], split["quarantined"]

    record_count(quarantined_jobs, "silver_quarantined_jobs")

    jobs_with_tech = backfill_technician_name(valid_jobs)
    jobs_with_dates = type_dates_and_derive_duration(jobs_with_tech)

    enriched = build_enriched_service_jobs(jobs_with_dates, bronze_customers, bronze_devices)

    record_count(enriched, "silver_enriched_service_jobs")

    
    quarantine_target = f"{CONFIG.silver_path}/{CONFIG.silver_quarantine_tbl}"
    (
        quarantined_jobs.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(quarantine_target)
    )

    enriched_target = f"{CONFIG.silver_path}/{CONFIG.silver_enriched_tbl}"
    (
        enriched.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("job_status")
        .save(enriched_target)
    )

    return {"enriched": enriched, "quarantined": quarantined_jobs}


silver_tables = run_silver_layer(bronze_customers_df, bronze_devices_df, bronze_service_jobs_df)
silver_enriched_df = silver_tables["enriched"]

2026-07-15 06:31:47,242 | INFO     | servicetrack | START stage='silver_transformation'
2026-07-15 06:31:48,259 | INFO     | servicetrack | [silver_quarantined_jobs] record count = 0
2026-07-15 06:31:50,006 | INFO     | servicetrack | [silver_enriched_service_jobs] record count = 1,500
2026-07-15 06:31:58,197 | INFO     | servicetrack | SUCCESS stage='silver_transformation' elapsed=10.95s


##### Note on partitioning
The enriched Silver table is written `partitionBy("job_status")` - only four
distinct values (`Completed` / `In Progress` / `Pending` / `Cancelled`), so this
avoids the classic "small files from over-partitioning" trap (partitioning by
`job_id`, which is unique per row, would be a correctness-neutral but
performance-destroying mistake). Gold-layer aggregations that filter or group
on `job_status` (SLA compliance, technician completed-jobs) benefit from
partition pruning; `device_brand` and `technician_id` - the other two
frequently-filtered columns - are handled via `ZORDER` instead of partitioning
(Section 7), since they have too many distinct values to partition on cleanly
without fragmenting the table into hundreds of tiny files.

## 6. Gold Layer - Business-Ready Aggregates
Three Gold tables, each answering a specific stakeholder question. All three
read from the single Silver `enriched_service_jobs` table - no re-joining
against Bronze - which is the point of Silver: pay the join cost once, reuse
the result for every downstream aggregate.

### 6.1 SLA_Performance_Gold
Per device brand + device type: total jobs, completed jobs, on-time vs delayed
(SLA = `CONFIG.sla_days`, default 5), and SLA compliance %. On-time/delayed are
only meaningful for **completed** jobs - an in-progress job hasn't missed
anything yet - so both counts are computed as conditional aggregates scoped to
`job_status == 'Completed'`.

In [0]:

def build_sla_performance_gold(enriched: DataFrame) -> DataFrame:
    is_completed = F.col("job_status") == F.lit("Completed")
    is_on_time = is_completed & (F.col("repair_duration_days") <= CONFIG.sla_days)
    is_delayed = is_completed & (F.col("repair_duration_days") > CONFIG.sla_days)

    sla_gold = (
        enriched.groupBy("device_brand", "device_type")
        .agg(
            F.count("*").alias("total_jobs"),
            F.sum(F.when(is_completed, 1).otherwise(0)).alias("completed_jobs"),
            F.sum(F.when(is_on_time, 1).otherwise(0)).alias("on_time_jobs"),
            F.sum(F.when(is_delayed, 1).otherwise(0)).alias("delayed_jobs"),
        )
        .withColumn(
            "sla_compliance_percentage",
            F.when(
                F.col("completed_jobs") > 0,
                F.round((F.col("on_time_jobs") / F.col("completed_jobs")) * 100, 2),
            ).otherwise(F.lit(0.0)),
        )
    )
    return sla_gold


### 6.2 Technician_Efficiency_Gold
Per technician: average repair duration, count of completed repairs, total
revenue, and average repair cost. All cost/duration metrics are scoped to
`Completed` jobs only - `actual_cost` and `repair_duration_days` are NULL for
anything else, so this is both a correctness requirement and (courtesy of
Spark's NULL-skipping aggregate functions) what `avg()` / `sum()` do by default.

In [0]:

def build_technician_efficiency_gold(enriched: DataFrame) -> DataFrame:
    completed_jobs = enriched.filter(F.col("job_status") == "Completed")

    technician_gold = (
        completed_jobs.groupBy("technician_id", "technician_name")
        .agg(
            F.round(F.avg("repair_duration_days"), 2).alias("average_repair_duration"),
            F.count("*").alias("completed_repairs"),
            F.round(F.sum("actual_cost"), 2).alias("total_revenue"),
            F.round(F.avg("actual_cost"), 2).alias("average_repair_cost"),
        )
        .orderBy(F.col("average_repair_duration").asc())
    )
    return technician_gold


### 6.3 Failure_Trends_Gold
Per device brand + model series: repair volume and total repair cost. This
table intentionally includes **all** job statuses (not just Completed) -
`repair_volume` is meant to show total *demand* on a device/model, a signal
for spare-parts procurement regardless of whether every job has closed out
yet. `total_repair_cost` naturally only reflects jobs that have an
`actual_cost` (i.e. completed ones), since `F.sum` skips NULLs.

In [0]:

def build_failure_trends_gold(enriched: DataFrame) -> DataFrame:
    failure_gold = (
        enriched.groupBy("device_brand", "model_series")
        .agg(
            F.count("*").alias("repair_volume"),
            F.round(F.sum("actual_cost"), 2).alias("total_repair_cost"),
        )
        .orderBy(F.col("repair_volume").desc())
    )
    return failure_gold


### 6.4 Orchestrate Gold + write + register in metastore

In [0]:
@run_stage("gold_aggregation")
def run_gold_layer(enriched: DataFrame) -> Dict[str, DataFrame]:
    sla_gold = build_sla_performance_gold(enriched)
    technician_gold = build_technician_efficiency_gold(enriched)
    failure_gold = build_failure_trends_gold(enriched)

    record_count(sla_gold, "gold_sla_performance")
    record_count(technician_gold, "gold_technician_efficiency")
    record_count(failure_gold, "gold_failure_trends")

    gold_specs = [
        (sla_gold, CONFIG.gold_sla_tbl),
        (technician_gold, CONFIG.gold_technician_tbl),
        (failure_gold, CONFIG.gold_failure_tbl),
    ]

    for df, tbl_name in gold_specs:
        target_path = f"{CONFIG.gold_path}/{tbl_name}"
        (
            df.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(target_path)
        )

    return {"sla": sla_gold, "technician": technician_gold, "failure": failure_gold}


gold_tables = run_gold_layer(silver_enriched_df)

2026-07-15 06:32:00,172 | INFO     | servicetrack | START stage='gold_aggregation'
2026-07-15 06:32:01,314 | INFO     | servicetrack | [gold_sla_performance] record count = 43
2026-07-15 06:32:03,124 | INFO     | servicetrack | [gold_technician_efficiency] record count = 8
2026-07-15 06:32:04,263 | INFO     | servicetrack | [gold_failure_trends] record count = 43
2026-07-15 06:32:13,771 | INFO     | servicetrack | SUCCESS stage='gold_aggregation' elapsed=13.6s


## 7. Optimizations

**Why `OPTIMIZE` + `ZORDER`?** Delta tables accumulate many small files across
writes (each streaming micro-batch or `overwrite` produces its own set of
Parquet files). `OPTIMIZE` compacts these into fewer, right-sized files, which
directly reduces the number of files the driver has to list and the number of
tasks Spark has to schedule on every downstream read. `ZORDER BY` goes
further: it co-locates rows with similar values of the Z-ORDER columns
*within* those compacted files, so a query filtering on `device_brand` or
`technician_id` can skip whole files via Delta's data-skipping statistics
instead of scanning the full table. We Z-ORDER the Silver enriched table (the
one everything else is built from, and the one analysts are most likely to
query ad hoc with a `WHERE device_brand = ...` or `WHERE technician_id = ...`
filter).

**Why `VACUUM`?** Delta's transaction log keeps old file versions around to
support time travel. Left unbounded, this is a storage-cost and file-count
leak. `VACUUM` physically deletes files no longer referenced by the log **and**
older than the retention window. We use the framework default of 168 hours
(7 days) - below that, Databricks requires explicitly disabling the
retention-duration safety check, which exists specifically to prevent someone
from accidentally deleting a file an in-flight time-travel query still needs.

In [0]:

@run_stage("optimization")
def run_optimizations():
    silver_path = f"{CONFIG.silver_path}/{CONFIG.silver_enriched_tbl}"

    silver_delta = DeltaTable.forPath(spark, silver_path)
    silver_delta.optimize().executeZOrderBy("device_brand", "technician_id")
    log.info(f"OPTIMIZE + ZORDER complete on {silver_path}")

    for tbl_name in [CONFIG.gold_sla_tbl, CONFIG.gold_technician_tbl, CONFIG.gold_failure_tbl]:
        gold_path = f"{CONFIG.gold_path}/{tbl_name}"
        DeltaTable.forPath(spark, gold_path).optimize().executeCompaction()
        log.info(f"OPTIMIZE (compaction) complete on {gold_path}")

    # VACUUM with a safe (>= 168h default) retention window on every managed table.
    for path in [silver_path] + [
        f"{CONFIG.gold_path}/{t}"
        for t in [CONFIG.gold_sla_tbl, CONFIG.gold_technician_tbl, CONFIG.gold_failure_tbl]
    ]:
        DeltaTable.forPath(spark, path).vacuum(retentionHours=CONFIG.vacuum_retention_hours)
        log.info(f"VACUUM complete on {path} (retention={CONFIG.vacuum_retention_hours}h)")


run_optimizations()

2026-07-15 06:32:14,082 | INFO     | servicetrack | START stage='optimization'
2026-07-15 06:32:18,142 | INFO     | servicetrack | OPTIMIZE + ZORDER complete on /Volumes/servicetrack/data/landing/delta/silver/enriched_service_jobs
2026-07-15 06:32:20,427 | INFO     | servicetrack | OPTIMIZE (compaction) complete on /Volumes/servicetrack/data/landing/delta/gold/sla_performance_gold
2026-07-15 06:32:22,475 | INFO     | servicetrack | OPTIMIZE (compaction) complete on /Volumes/servicetrack/data/landing/delta/gold/technician_efficiency_gold
2026-07-15 06:32:24,603 | INFO     | servicetrack | OPTIMIZE (compaction) complete on /Volumes/servicetrack/data/landing/delta/gold/failure_trends_gold
2026-07-15 06:32:36,241 | INFO     | servicetrack | VACUUM complete on /Volumes/servicetrack/data/landing/delta/silver/enriched_service_jobs (retention=168h)
2026-07-15 06:32:42,781 | INFO     | servicetrack | VACUUM complete on /Volumes/servicetrack/data/landing/delta/gold/sla_performance_gold (retentio

## 8. Validation - Data Quality Gate
A pipeline that writes Gold tables without checking them is a pipeline that
finds out about data quality bugs from an angry stakeholder instead of from
itself. This section runs a small set of hard assertions; any failure raises
and stops the notebook rather than letting bad Gold data ship silently.

In [0]:

def validate_pipeline(
    bronze_jobs: DataFrame,
    enriched: DataFrame,
    quarantined: DataFrame,
    gold: Dict[str, DataFrame],
) -> List[Dict[str, str]]:
    checks = []

    # 1. Row conservation: valid + quarantined (post de-dup) should equal the
    #    de-duplicated bronze row count.
    bronze_deduped_count = bronze_jobs.dropDuplicates().dropDuplicates(["job_id"]).count()
    silver_total = enriched.count() + quarantined.count()
    checks.append({
        "check": "row_conservation_bronze_to_silver",
        "passed": str(bronze_deduped_count == silver_total),
        "detail": f"bronze_deduped={bronze_deduped_count}, silver_total={silver_total}",
    })

    # 2. No NULL job_id / customer_id should remain in the enriched table.
    bad_keys = enriched.filter(F.col("job_id").isNull() | F.col("customer_id").isNull()).count()
    checks.append({
        "check": "no_null_keys_in_silver",
        "passed": str(bad_keys == 0),
        "detail": f"bad_key_rows={bad_keys}",
    })

    # 3. No duplicate job_id in the enriched table.
    dup_count = enriched.groupBy("job_id").count().filter(F.col("count") > 1).count()
    checks.append({
        "check": "no_duplicate_job_id_in_silver",
        "passed": str(dup_count == 0),
        "detail": f"duplicate_job_ids={dup_count}",
    })

    # 4. Gold SLA compliance percentages must fall within [0, 100].
    sla_out_of_range = (
        gold["sla"]
        .filter((F.col("sla_compliance_percentage") < 0) | (F.col("sla_compliance_percentage") > 100))
        .count()
    )
    checks.append({
        "check": "sla_percentage_within_bounds",
        "passed": str(sla_out_of_range == 0),
        "detail": f"out_of_range_rows={sla_out_of_range}",
    })

    # 5. Every technician in Gold must have a non-null name (backfill worked).
    null_tech_names = gold["technician"].filter(F.col("technician_name").isNull()).count()
    checks.append({
        "check": "technician_names_backfilled",
        "passed": str(null_tech_names == 0),
        "detail": f"null_technician_names={null_tech_names}",
    })

    return checks


@run_stage("validation")
def run_validation():
    results = validate_pipeline(
        bronze_service_jobs_df, silver_enriched_df, silver_tables["quarantined"], gold_tables
    )
    failed = [r for r in results if r["passed"] == "False"]
    for r in results:
        status = "PASS" if r["passed"] == "True" else "FAIL"
        log.info(f"[{status}] {r['check']} - {r['detail']}")

    if failed:
        raise AssertionError(f"{len(failed)} validation check(s) failed: {failed}")

    log.info("All validation checks passed.")
    return results


validation_results = run_validation()

2026-07-15 06:32:55,696 | INFO     | servicetrack | START stage='validation'
2026-07-15 06:33:02,677 | INFO     | servicetrack | [PASS] row_conservation_bronze_to_silver - bronze_deduped=1500, silver_total=1500
2026-07-15 06:33:02,677 | INFO     | servicetrack | [PASS] no_null_keys_in_silver - bad_key_rows=0
2026-07-15 06:33:02,678 | INFO     | servicetrack | [PASS] no_duplicate_job_id_in_silver - duplicate_job_ids=0
2026-07-15 06:33:02,678 | INFO     | servicetrack | [PASS] sla_percentage_within_bounds - out_of_range_rows=0
2026-07-15 06:33:02,679 | INFO     | servicetrack | [PASS] technician_names_backfilled - null_technician_names=0
2026-07-15 06:33:02,679 | INFO     | servicetrack | All validation checks passed.
2026-07-15 06:33:02,679 | INFO     | servicetrack | SUCCESS stage='validation' elapsed=6.98s


## 9. Final Summary

### Time travel example
Because Delta Lake logs every write as a numbered version, the Silver table
can be queried exactly as it looked before today's run - useful for debugging
a Gold-layer regression without re-running the whole pipeline.

In [0]:
silver_path = f"{CONFIG.silver_path}/{CONFIG.silver_enriched_tbl}"

# Version-based time travel (version 0 = first write)
try:
    df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(silver_path)
    log.info(f"Time travel OK - version 0 of enriched_service_jobs has {df_v0.count():,} rows")
except Exception as exc:  # first run may only have one version
    log.info(f"Time travel to version 0 skipped: {exc}")

2026-07-15 06:33:04,089 | INFO     | servicetrack | Time travel OK - version 0 of enriched_service_jobs has 1,500 rows


### DESCRIBE HISTORY example
Every Delta write is auditable: who ran it, when, what operation, how many
rows changed. This is the audit trail a plain Parquet table cannot give you.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY delta.`{silver_path}`").select(  # noqa: F821
    "version", "timestamp", "operation", "operationMetrics"
))

version,timestamp,operation,operationMetrics
5,2026-07-15T06:31:58.000Z,WRITE,"Map(numFiles -> 4, numRemovedFiles -> 4, numRemovedBytes -> 105326, numDeletionVectorsRemoved -> 0, numOutputRows -> 1500, numOutputBytes -> 105326)"
4,2026-07-14T18:22:52.000Z,WRITE,"Map(numFiles -> 4, numRemovedFiles -> 4, numRemovedBytes -> 105534, numDeletionVectorsRemoved -> 0, numOutputRows -> 1500, numOutputBytes -> 105326)"
3,2026-07-14T18:05:04.000Z,WRITE,"Map(numFiles -> 4, numRemovedFiles -> 4, numRemovedBytes -> 105326, numDeletionVectorsRemoved -> 0, numOutputRows -> 1500, numOutputBytes -> 105534)"
2,2026-07-14T16:59:25.000Z,WRITE,"Map(numFiles -> 4, numRemovedFiles -> 4, numRemovedBytes -> 105326, numDeletionVectorsRemoved -> 0, numOutputRows -> 1500, numOutputBytes -> 105326)"
1,2026-07-14T14:51:46.000Z,WRITE,"Map(numFiles -> 4, numRemovedFiles -> 4, numRemovedBytes -> 105326, numDeletionVectorsRemoved -> 0, numOutputRows -> 1500, numOutputBytes -> 105326)"
0,2026-07-14T14:35:49.000Z,WRITE,"Map(numFiles -> 4, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1500, numOutputBytes -> 105326)"


### Pipeline run summary

In [0]:
summary = {
    "environment": CONFIG.env,
    "sla_threshold_days": CONFIG.sla_days,
    "bronze_service_jobs_rows": bronze_service_jobs_df.count(),
    "silver_enriched_rows": silver_enriched_df.count(),
    "silver_quarantined_rows": silver_tables["quarantined"].count(),
    "gold_sla_performance_rows": gold_tables["sla"].count(),
    "gold_technician_efficiency_rows": gold_tables["technician"].count(),
    "gold_failure_trends_rows": gold_tables["failure"].count(),
    "validation_checks_passed": all(r["passed"] == "True" for r in validation_results),
}

log.info("=" * 60)
log.info("SERVICETRACK PIPELINE - RUN SUMMARY")
log.info("=" * 60)
for k, v in summary.items():
    log.info(f"{k:35s}: {v}")
log.info("=" * 60)

display(spark.createDataFrame([summary]))  # noqa: F821

2026-07-15 06:33:18,725 | INFO     | servicetrack | ============================================================
2026-07-15 06:33:18,726 | INFO     | servicetrack | SERVICETRACK PIPELINE - RUN SUMMARY
2026-07-15 06:33:18,726 | INFO     | servicetrack | ============================================================
2026-07-15 06:33:18,727 | INFO     | servicetrack | environment                        : dev
2026-07-15 06:33:18,728 | INFO     | servicetrack | sla_threshold_days                 : 5
2026-07-15 06:33:18,729 | INFO     | servicetrack | bronze_service_jobs_rows           : 1510
2026-07-15 06:33:18,730 | INFO     | servicetrack | silver_enriched_rows               : 1500
2026-07-15 06:33:18,730 | INFO     | servicetrack | silver_quarantined_rows            : 0
2026-07-15 06:33:18,731 | INFO     | servicetrack | gold_sla_performance_rows          : 43
2026-07-15 06:33:18,732 | INFO     | servicetrack | gold_technician_efficiency_rows    : 8
2026-07-15 06:33:18,732 | INFO     | ser

bronze_service_jobs_rows,environment,gold_failure_trends_rows,gold_sla_performance_rows,gold_technician_efficiency_rows,silver_enriched_rows,silver_quarantined_rows,sla_threshold_days,validation_checks_passed
1510,dev,43,43,8,1500,0,5,true


### Sample Gold outputs

In [0]:
display(gold_tables["sla"].orderBy(F.col("sla_compliance_percentage").asc()))  # noqa: F821

device_brand,device_type,total_jobs,completed_jobs,on_time_jobs,delayed_jobs,sla_compliance_percentage
Realme,Refrigerator,30,21,10,11,47.62
Dell,Refrigerator,28,19,11,8,57.89
Acer,Tablet,38,25,15,10,60.0
OnePlus,Washing Machine,44,31,19,12,61.29
Samsung,Smart TV,29,19,12,7,63.16
Acer,Monitor,32,23,15,8,65.22
Lenovo,Microwave,34,23,15,8,65.22
Apple,Laptop,36,23,15,8,65.22
HP,Monitor,38,29,19,10,65.52
LG,Tablet,34,29,19,10,65.52


In [0]:
display(gold_tables["technician"])  # noqa: F821

technician_id,technician_name,average_repair_duration,completed_repairs,total_revenue,average_repair_cost
T005,Kavitha Nair,2.57,143,619916.39,4335.08
T001,Rajesh Kumar,3.0,132,588641.21,4459.40
T003,Priya Singh,3.52,140,644785.17,4605.61
T007,Meena Reddy,4.19,152,711123.01,4678.44
T002,Suresh Rao,4.32,122,523377.69,4289.98
T006,Deepak Sharma,5.31,135,606156.74,4490.05
T004,Amit Patel,6.4,119,520505.27,4373.99
T008,Arjun Iyer,7.05,111,527214.74,4749.68


In [0]:
display(gold_tables["failure"].limit(10))  # noqa: F821

device_brand,model_series,repair_volume,total_repair_cost
Oppo,Oppo WAS-Series,47,133903.94
OnePlus,OnePlus WAS-Series,44,126050.57
Sony,Sony MON-Series,43,149876.10
Vivo,Vivo PRI-Series,42,127002.92
OnePlus,OnePlus MON-Series,42,179972.67
Lenovo,Lenovo MON-Series,42,125706.55
Motorola,Motorola PRI-Series,42,169105.14
Oppo,Oppo MIC-Series,41,122204.88
LG,LG SMA-Series,39,128307.81
Lenovo,Lenovo SMA-Series,38,91935.89


## Appendix: Catalyst Optimizer, Delta Lake, and why this design scales

- **Catalyst Optimizer.** Every transformation in this notebook (`.filter`,
  `.join`, `.groupBy`, `.withColumn`) builds a *logical plan* - it does not
  execute anything. Catalyst rewrites that logical plan (predicate pushdown,
  column pruning, constant folding, join reordering) before Spark's
  cost-based optimizer picks a physical plan, and only then does any data
  move. Concretely: the `job_status == "Completed"` filter inside
  `build_technician_efficiency_gold` gets pushed down and combined with the
  Delta file-skipping statistics from Section 7's `ZORDER`, so files that
  contain no completed jobs for the technicians in scope are never even read.
- **Delta Lake ACID + schema enforcement.** If `run_gold_layer` fails halfway
  through writing the three Gold tables, Delta's transaction log guarantees no
  table is left half-written - a reader either sees the old version or the
  fully-committed new version, never a partial one. Schema enforcement means a
  malformed upstream export that drops a column fails loudly at write time
  instead of silently producing a Gold table with a NULL-filled column.
- **Why this stays correct at 100x the volume.** Nothing in this pipeline's
  *logic* depends on file count or row count - Auto Loader's incremental
  listing, the broadcast-join pattern (always broadcast the dimension, never
  the fact), the ZORDER-not-partition choice for high-cardinality columns, and
  the quarantine-not-drop invalid-record policy are all volume-independent
  design decisions. Going from 1,510 service jobs a quarter to 5 million a
  quarter changes cluster sizing, not this notebook's code.

In [0]:
%sql
-- Repeat Customer Detection
SELECT 
    customer_id, 
    customer_name, 
    device_brand, 
    issue_type, 
    COUNT(job_id) AS repeat_visit_count
FROM delta.`/Volumes/servicetrack/data/landing/delta/silver/enriched_service_jobs`
GROUP BY 
    customer_id, 
    customer_name, 
    device_brand, 
    issue_type
HAVING COUNT(job_id) > 1
ORDER BY repeat_visit_count DESC;

customer_id,customer_name,device_brand,issue_type,repeat_visit_count
CUST0231,Neha Fernandez,Samsung,Charging Port Fault,2
CUST0095,Sachin Nambiar,Oppo,Hinge Broken,2
CUST0035,Gaurav Chaudhary,OnePlus,Wi-Fi Not Connecting,2
CUST0075,Meera Shetty,Lenovo,Keyboard Fault,2
CUST0088,Sushma D'Souza,Xiaomi,Speaker Problem,2
CUST0035,Gaurav Chaudhary,Motorola,Wi-Fi Not Connecting,2
CUST0142,Yash Pandey,Acer,RAM Issue,2
CUST0028,Tushar Patel,Lenovo,Water Damage,2
CUST0253,Kavya Mishra,Dell,Charging Port Fault,2
CUST0167,Aditya D'Souza,Acer,Overheating,2


In [0]:
%sql
-- Extract the most recent job per customer using CTEs and Window Functions
WITH RankedCustomerVisits AS (
    SELECT 
        customer_id,
        customer_name,
        job_id,
        issue_type,
        job_status,
        received_date,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY received_date DESC) as visit_rank
    FROM delta.`/Volumes/servicetrack/data/landing/delta/silver/enriched_service_jobs`
)
SELECT 
    customer_id,
    customer_name,
    job_id AS most_recent_job_id,
    issue_type,
    job_status,
    received_date AS last_visit_date
FROM RankedCustomerVisits
WHERE visit_rank = 1
ORDER BY last_visit_date DESC;


customer_id,customer_name,most_recent_job_id,issue_type,job_status,last_visit_date
CUST0064,Imran Singh,JOB00467,Power Not Turning On,In Progress,2024-03-21T00:00:00.000Z
CUST0082,Madhu Verma,JOB01310,Motherboard Failure,In Progress,2024-03-21T00:00:00.000Z
CUST0094,Sanjay Bajaj,JOB01480,Power Not Turning On,Completed,2024-03-21T00:00:00.000Z
CUST0095,Sachin Nambiar,JOB01119,Hinge Broken,In Progress,2024-03-21T00:00:00.000Z
CUST0099,Tanvi Iyer,JOB00655,Wi-Fi Not Connecting,Cancelled,2024-03-21T00:00:00.000Z
CUST0121,Nisha Bajaj,JOB00101,Camera Not Working,Completed,2024-03-21T00:00:00.000Z
CUST0125,Ajay Ali,JOB00442,Screen Damage,Completed,2024-03-21T00:00:00.000Z
CUST0130,Riya Gupta,JOB00530,Camera Not Working,Completed,2024-03-21T00:00:00.000Z
CUST0156,Simran Mehta,JOB00988,Software Crash,Completed,2024-03-21T00:00:00.000Z
CUST0183,Lakshmi Khan,JOB00627,Battery Issue,Completed,2024-03-21T00:00:00.000Z


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, current_date

# 1. Define the path for our new Technician Dimension table
tech_dim_path = f"{CONFIG.gold_path}/technician_dim_scd2"

# 2. Extract the initial list of technicians from Silver and add SCD2 tracking columns
tech_dim_initial = (
    silver_enriched_df.select("technician_id", "technician_name")
    .filter(col("technician_id").isNotNull())
    .dropDuplicates(["technician_id"])
    .withColumn("is_active", lit(True))
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast("date"))
)

# Write the initial table to Gold
tech_dim_initial.write.format("delta").mode("overwrite").save(tech_dim_path)

# 3. Simulate an update: Technician T001 gets promoted and updates their name
updates_df = spark.createDataFrame(
    [("T001", "Rajesh Kumar (Senior Tech)")], 
    ["technician_id", "technician_name"]
)

# 4. Perform the SCD Type 2 MERGE operation
target_delta = DeltaTable.forPath(spark, tech_dim_path)

# Prepare the staged data (identifying which existing rows need to be retired)
staged_updates = (
    updates_df.alias("updates")
    .join(target_delta.toDF().alias("target"), "technician_id")
    .where("target.is_active = true AND target.technician_name <> updates.technician_name")
    .selectExpr("updates.technician_id", "updates.technician_name", "'mergeKey' as mergeKey")
)

# Union the retiring rows with the new incoming active rows
staged_all = staged_updates.union(
    updates_df.selectExpr("technician_id", "technician_name", "technician_id as mergeKey")
)

# Execute the Delta Merge
target_delta.alias("target").merge(
    staged_all.alias("staged"),
    "target.technician_id = staged.mergeKey"
).whenMatchedUpdate(
    condition="target.is_active = true AND target.technician_name <> staged.technician_name",
    set={
        "is_active": lit(False),
        "end_date": current_date()
    }
).whenNotMatchedInsert(
    values={
        "technician_id": "staged.technician_id",
        "technician_name": "staged.technician_name",
        "is_active": lit(True),
        "start_date": current_date(),
        "end_date": lit(None).cast("date")
    }
).execute()

# 5. Display the result to prove the history is tracked!
print("SCD Type 2 Merge Complete. Showing history for T001:")
display(spark.read.format("delta").load(tech_dim_path).filter("technician_id = 'T001'"))

SCD Type 2 Merge Complete. Showing history for T001:


technician_id,technician_name,is_active,start_date,end_date
T001,Rajesh Kumar (Senior Tech),true,2026-07-15,null
T001,Rajesh Kumar,false,2026-07-15,2026-07-15


In [0]:
# Register Gold tables to the Databricks Catalog
spark.read.format("delta").load(f"{CONFIG.gold_path}/technician_efficiency_gold") \
    .write.mode("overwrite").saveAsTable("default.technician_efficiency_gold")

spark.read.format("delta").load(f"{CONFIG.gold_path}/failure_trends_gold") \
    .write.mode("overwrite").saveAsTable("default.failure_trends_gold")

spark.read.format("delta").load(f"{CONFIG.gold_path}/sla_performance_gold") \
    .write.mode("overwrite").saveAsTable("default.sla_performance_gold")

print("Tables successfully registered to the 'default' database!")

Tables successfully registered to the 'default' database!
